# Payment Reconciliation Automation

> Synthetic reconstruction of a real-world payment reconciliation workflow.  
> No proprietary company data, code, credentials, or confidential business logic is included.

---

## 1. Setup

In [ ]:
from pathlib import Path
import sys

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import (
    DOWNLOAD_DIR,
    UPLOAD_DIR,
    OUTPUT_DIR,
    BANK_A_FILE_PREFIX,
    BANK_B_FILE_PREFIX,
    API_ENDPOINT,
    INGESTION_TOKEN,
    INGESTION_PARENT_DIR,
)

from src.utils import get_source_files
from src.bank_a import process_bank_a
from src.bank_b import process_bank_b
from src.export import export_results
from src.ingestion import upload_file

---
## 2. Prepare Input Files
The workflow processes transaction statements received from multiple banking sources.

In [ ]:
bank_a_files = get_source_files(
    DOWNLOAD_DIR,
    BANK_A_FILE_PREFIX,
)

bank_b_files = get_source_files(
    DOWNLOAD_DIR,
    BANK_B_FILE_PREFIX,
)

print(f"Bank A files: {len(bank_a_files)}")
print(f"Bank B files: {len(bank_b_files)}")

---

## 3. Process Bank A Statements
Bank-specific processing includes:

- detecting the actual data header
- standardizing columns
- cleaning dates and amounts
- extracting transaction references
- classifying transactions

In [ ]:
bank_a_results = process_bank_a(
    bank_a_files
)

In [ ]:
# Inspect the output
for transaction_type, df in bank_a_results.items():
    print(
        f"{transaction_type}: "
        f"{len(df):,} rows"
    )

---

## 4. Process Bank B Statements
Bank B has a different statement structure and transaction-detail format, so it uses a separate processing module.

In [ ]:
bank_b_results = process_bank_b(
    bank_b_files
)

In [ ]:
# Inspect the output
for transaction_type, df in bank_b_results.items():
    print(
        f"{transaction_type}: "
        f"{len(df):,} rows"
    )

---

## 5. Export Processed Data
Processed datasets are exported as standardized CSV files.

In [ ]:
bank_a_outputs = export_results(
    results=bank_a_results,
    output_dir=OUTPUT_DIR,
    file_prefix="BANK_A",
)

In [ ]:
bank_b_outputs = export_results(
    results=bank_b_results,
    output_dir=OUTPUT_DIR,
    file_prefix="BANK_B",
)

In [ ]:
# Review generated files:

print("Bank A outputs:")
for file in bank_a_outputs:
    print(file)

print("\nBank B outputs:")
for file in bank_b_outputs:
    print(file)

---

## 6. Upload to Data Ingestion Layer
In production, the processed files can be uploaded to a downstream data platform through an ingestion API.

For this public repository, the API configuration is synthetic and should not contain real credentials.

In [ ]:
for file in bank_a_outputs + bank_b_outputs:

    upload_file(
        api_endpoint=API_ENDPOINT,
        ingestion_token=INGESTION_TOKEN,
        file_path=file,
        parent_dir=INGESTION_PARENT_DIR,
    )

> The upload step is included to demonstrate the production-style architecture.  
> The public repository does not connect to any real company or banking system.


---

## 7. Next Step: SQL Reconciliation

After ingestion, the processed transaction data is available in the downstream data layer (DataHub).

The reconciliation itself is performed using SQL, organized as five controls across the disbursement and repayment lifecycle. Controls 3 and 5 are each split into two stages, since a funding/settlement module sits between the internal transaction record and the external bank statement and is used as the key bridge (direct or batch/grouped) between the two:

```text
Control 1   Source / upstream            <> Transaction state (confirm)
Control 2   Transaction state (confirm)  <> Transaction state (disburse)
Control 3A  Transaction state (disburse) <> Funding (settlement) module
Control 3B  Funding (settlement) module  <> Bank statement
Control 4   Payment                      <> Transaction state (repay)
Control 5A  Transaction state (repay)    <> Funding (settlement) module
Control 5B  Funding (settlement) module  <> Bank statement
```

Each control follows the same pattern: independent daily totals per side, a `FULL OUTER JOIN` reconciliation pool, timing-difference detection, and a known-issue mapping (sourced from an Ops-maintained Google Sheet also ingested into DataHub) — leaving only the unexplained cases in the RCA queue. The funding-module-to-bank-statement controls (3B / 5B) also carry the funding module's latest status for that order.

```text
Source Bank Statements
        ↓
Python Processing
        ↓
Standardized Transaction Files
        ↓
Data Ingestion API
        ↓
DataHub Staging Tables
        ↓
SQL Reconciliation (5 controls, 7 checks)
        ↓
Timing / Known-Issue Classification
        ↓
RCA / Manual Investigation
```

See:

```text
sql/
├── recon_01/   Control 1  - Source -> Transaction (confirm)
├── recon_02/   Control 2  - Transaction (confirm) -> Transaction (disburse)
├── recon_03/   Control 3A - Transaction (disburse) -> Funding module
│               Control 3B - Funding module -> Bank statement
├── recon_04/   Control 4  - Payment -> Transaction (repay)
└── recon_05/   Control 5A - Transaction (repay) -> Funding module
                Control 5B - Funding module -> Bank statement
```

Each `recon_NN/` folder contains the daily summary queries plus one or more `discrepancy*.sql` files that produce the RCA-ready output. The orchestrator that runs all of them on a schedule and publishes results to Google Sheets is `src/daily_reconciliation.py`.

---

## Workflow Summary


    Payment Reconciliation Workflow

    1. Read source bank statements
    2. Clean and standardize data
    3. Extract transaction references
    4. Classify transactions
    5. Export standardized datasets
    6. Upload to the data ingestion layer (DataHub)
    7. Run the SQL reconciliation controls (src/daily_reconciliation.py)
    8. Classify discrepancies (timing vs. known issue vs. unresolved)
    9. Publish summary + RCA outputs to Google Sheets
